# Lab Assignment 4

MST Dependency Parsing and Training

**Total: 20 marks**

## Objective
Implement core components of an MST dependency parser, analyse a Chu-Liu/Edmonds cycle, and perform one structured-perceptron update using gold and predicted dependency trees.

## Submission
- Complete all `TODO` sections and all markdown answers.
- Do not modify the supplied data or expected-test cells.
- Run **Kernel > Restart & Run All** before submitting the completed `.ipynb` file.

## Valid dependency tree conditions
Each non-root word has exactly one incoming head; `ROOT` has none; all words are reachable from `ROOT`; and no directed cycle occurs.

In [1]:
from itertools import product
from collections import defaultdict, Counter
import pandas as pd
import matplotlib.pyplot as plt

## Task 1: Validate Dependency Trees (2 marks)

Implement `is_valid_tree()` to verify whether a set of dependency arcs forms a valid rooted dependency tree.

A valid dependency tree must satisfy all of the following conditions:

1. `ROOT` has no incoming arc.
2. Every non-root word has exactly one incoming head.
3. No word can be its own head (no self-loop).
4. The graph must not contain a directed cycle.
5. Every word must be reachable from `ROOT`.
6. The number of arcs must be exactly `number_of_tokens - 1`.

### Your task

Complete the `is_valid_tree()` function below.

Do not change the test cases.

Your function should return:

- `True` for a valid dependency tree.
- `False` for an invalid dependency structure.

You should ensure that the function correctly detects duplicate heads, missing heads, cycles, self-loops, disconnected components, and arcs entering `ROOT`.

In [2]:
def is_valid_tree(tokens, arcs, root="ROOT"):
    """
    Check whether `arcs` forms a valid rooted dependency tree over `tokens`.

    Conditions checked (see markdown above):
      1. ROOT has no incoming arc.
      2. Every non-root word has exactly one incoming head.
      3. No word is its own head.
      4. No directed cycle.
      5. Every word is reachable from ROOT.
      6. len(arcs) == number_of_tokens - 1.
    """
    non_root = [w for w in tokens if w != root]

    # Condition 6: exact number of arcs.
    if len(arcs) != len(non_root):
        return False

    heads = {}
    for head, dep in arcs:
        # Condition 1: nothing may enter ROOT.
        if dep == root:
            return False
        # Condition 3: no self-loop.
        if head == dep:
            return False
        # Condition 2 (duplicate heads): a dependent may appear at most once.
        if dep in heads:
            return False
        heads[dep] = head

    # Condition 2 (missing heads): every non-root word must have a head.
    if set(heads.keys()) != set(non_root):
        return False

    # Conditions 4 & 5: no cycle, and every word reachable from ROOT.
    for start in non_root:
        current = start
        seen = set()
        while current != root:
            if current in seen:
                return False  # directed cycle
            if current not in heads:
                return False  # cannot reach ROOT
            seen.add(current)
            current = heads[current]

    return True

test_tokens = ["ROOT", "A", "B", "C"]
valid = [("ROOT","A"), ("A","B"), ("A","C")]
cycle = [("ROOT","A"), ("B","C"), ("C","B")]
duplicate = [("ROOT","A"), ("A","B"), ("C","B")]
print(is_valid_tree(test_tokens, valid))      # Expected: True
print(is_valid_tree(test_tokens, cycle))      # Expected: False
print(is_valid_tree(test_tokens, duplicate))  # Expected: False

True
False
False


## Task 2: Exhaustive MST Reference Parser (3 marks)

Implement an exhaustive reference parser for a small dependency graph.

For every non-root word:

1. Enumerate all possible incoming heads.
2. Construct candidate dependency trees.
3. Use `is_valid_tree()` from Task 1 to reject invalid structures.
4. Compute the total score of every valid tree.
5. Return the highest-scoring tree and its score.

The score of a tree is:

$$
Score(G)=\sum_{(h,d)\in G} Score(h\rightarrow d)
$$

where:

- $h$ = head
- $d$ = dependent
- $G$ = dependency tree

### Your task

Complete:

```python
exhaustive_mst(tokens, score)

In [3]:

tokens = ["ROOT", "Researchers", "analyze", "data", "carefully"]

score = {
    ("ROOT","Researchers"):1,
    ("ROOT","analyze"):7,
    ("ROOT","data"):1,
    ("ROOT","carefully"):1,

    ("Researchers","analyze"):2,
    ("Researchers","data"):2,
    ("Researchers","carefully"):2,

    ("analyze","Researchers"):6,
    ("analyze","data"):7,
    ("analyze","carefully"):6,

    ("data","Researchers"):2,
    ("data","analyze"):3,
    ("data","carefully"):3,

    ("carefully","Researchers"):2,
    ("carefully","analyze"):3,
    ("carefully","data"):4,
}

def exhaustive_mst(tokens, score, root="ROOT"):
    """
    Brute-force reference parser: try every combination of heads for the
    non-root words, keep only the valid trees, and return the one with the
    highest total score.
    """
    non_root = [w for w in tokens if w != root]

    # 1. Enumerate all possible incoming heads for every non-root word.
    choices = [[h for h in tokens if h != dep] for dep in non_root]

    best_score = float("-inf")
    best_tree = None

    # 2. Construct every candidate tree.
    for heads in product(*choices):
        tree = list(zip(heads, non_root))

        # 3. Reject invalid structures using Task 1's checker.
        if not is_valid_tree(tokens, tree, root):
            continue

        # 4. Compute the total score of this valid tree.
        total = sum(score[a] for a in tree)

        # 5. Keep the best-scoring valid tree.
        if total > best_score:
            best_score = total
            best_tree = tree

    return best_score, best_tree

best_score, best_arcs = exhaustive_mst(tokens, score)

print("Best score:", best_score)
print("Best arcs:", best_arcs)

Best score: 26
Best arcs: [('analyze', 'Researchers'), ('ROOT', 'analyze'), ('analyze', 'data'), ('analyze', 'carefully')]


## Task 3: Chu-Liu/Edmonds — Best Incoming Edges and Cycle Detection (3 marks)

The Chu-Liu/Edmonds algorithm starts by selecting the highest-scoring incoming edge for every non-root node.

For each node $v$:

$$
parent(v)
=
\arg\max_{u\neq v} Score(u\rightarrow v)
$$

If the selected edges form a valid tree, the algorithm is finished.

However, if the selected edges contain a directed cycle, the cycle must be detected and contracted before continuing.

### Your task

Complete the following two functions:

1. `best_incoming()`
2. `find_cycle()`

`best_incoming()` should select the highest-scoring incoming edge for every non-root node.

`find_cycle()` should:

- return one directed cycle as a list of nodes if a cycle exists;
- return `None` if no cycle exists.

### Questions

After running the code:

1. What edges were selected by the greedy incoming-edge step?
2. Which nodes form the cycle?
3. Why is the greedy result not a valid dependency tree?
4. Why can we not simply accept the greedy result as the MST?

In [4]:
cycle_tokens = ["ROOT", "A", "B", "C"]

cycle_score = {
    ("ROOT","A"):5,
    ("ROOT","B"):4,
    ("ROOT","C"):3,

    ("A","B"):10,
    ("B","A"):10,

    ("A","C"):7,
    ("B","C"):8,

    ("C","A"):2,
    ("C","B"):2,
}


def best_incoming(tokens, score, root="ROOT"):
    """
    For every non-root node, select the incoming edge with the highest
    score. Edges into ROOT are never considered (ROOT is never a
    dependent). Returns a list of (head, dependent) tuples.
    """
    arcs = []
    for dep in tokens:
        if dep == root:
            continue
        candidates = [
            (head, score[(head, dep)])
            for head in tokens
            if head != dep and (head, dep) in score
        ]
        if not candidates:
            continue
        best_head, _ = max(candidates, key=lambda pair: pair[1])
        arcs.append((best_head, dep))
    return arcs


def find_cycle(arcs):
    """
    Follow parent (head) pointers starting from every node; if we revisit
    a node while following one chain, the nodes from the first occurrence
    of that node onward form a directed cycle. Returns None if no cycle
    is found.
    """
    head_of = {dep: head for head, dep in arcs}

    visited = set()
    for start in head_of:
        if start in visited:
            continue
        path = []
        seen = {}
        current = start
        while current in head_of and current not in visited:
            if current in seen:
                idx = path.index(current)
                return path[idx:]
            seen[current] = True
            path.append(current)
            current = head_of[current]
        visited.update(path)
    return None


greedy_arcs = best_incoming(cycle_tokens, cycle_score)

print("Greedy arcs:", greedy_arcs)
print("Cycle:", find_cycle(greedy_arcs))

Greedy arcs: [('B', 'A'), ('A', 'B'), ('B', 'C')]
Cycle: ['A', 'B']


**Task 3 explanation:**

Running the code above produces:

```
Greedy arcs: [('B', 'A'), ('A', 'B'), ('B', 'C')]
Cycle: ['A', 'B']
```

1. **Edges selected by the greedy incoming-edge step.** For every non-root node we picked the single highest-scoring incoming edge, independently of every other node: `B -> A` (score 10, beats `ROOT -> A`=5 and `C -> A`=2), `A -> B` (score 10, beats `ROOT -> B`=4 and `C -> B`=2), and `B -> C` (score 8, beats `ROOT -> C`=3 and `A -> C`=7).

2. **Nodes forming the cycle.** `A` and `B` form the cycle: `A -> B -> A`. Each chose the other as its best head, so following the parent pointers from either node loops forever between the two.

3. **Why the greedy result is not a valid dependency tree.** A valid dependency tree cannot contain a directed cycle (Task 1, condition 4) — every non-root word must be reachable from `ROOT` by following head arcs exactly once, and a cycle like `A <-> B` makes `A` and `B` unreachable from `ROOT` even though each individually has an incoming arc. `is_valid_tree` would reject this structure.

4. **Why we cannot simply accept the greedy result as the MST.** Selecting each node's best incoming edge independently maximizes each *local* choice, but it ignores the *global* tree constraint (no cycles, single root, full connectivity). The greedy edges here score `10 + 10 + 8 = 28` locally, but they don't form a spanning tree at all, so that number isn't even a valid tree score. This is exactly why Chu-Liu/Edmonds is needed: whenever the greedy step creates a cycle, the cycle must be contracted into a supernode and the problem solved recursively so the final structure is guaranteed to be an actual tree.

## Task 4: Chu-Liu/Edmonds Cycle Contraction (3 marks)

Suppose the greedy step produces the cycle:

$$
A\rightarrow B
$$

and

$$
B\rightarrow A
$$

with:

$$
Score(A\rightarrow B)=10
$$

$$
Score(B\rightarrow A)=10
$$

The total score of the selected cycle is:

$$
CycleScore=10+10=20
$$

When an external edge enters a node inside the cycle, the cycle is contracted into a single **supernode**.

For an external edge $h\rightarrow d$ entering the cycle, calculate the adjusted score as:

$$
\boxed{
AdjustedScore(h\rightarrow d)
=
Score(h\rightarrow d)
+
CycleScore
-
Score(old\_incoming\_edge(d))
}
$$

### Example

For:

$$
ROOT\rightarrow A
$$

we have:

$$
Score(ROOT\rightarrow A)=5
$$

The selected edge entering `A` inside the cycle is:

$$
B\rightarrow A
$$

with score:

$$
Score(B\rightarrow A)=10
$$

Therefore:

$$
AdjustedScore(ROOT\rightarrow A)
=
5+20-10
=
15
$$

### Your task

Complete `entering_cycle_table()`.

Create a table for:

- `ROOT -> A`
- `ROOT -> B`

The table must contain:

- candidate edge
- original score
- selected cycle edge being removed
- removed edge score
- total cycle score
- adjusted score

### Interpretation question

Which candidate edge should be selected to enter the contracted cycle?

Explain which original cycle edge will be removed when the contracted tree is expanded back to the original graph.

In [5]:
def entering_cycle_table(score, cycle, selected_cycle_arcs, root="ROOT"):
    """
    Build a DataFrame describing, for every edge (h, d) that enters the
    cycle from outside (h not in cycle, d in cycle), the adjusted score
    used during Chu-Liu/Edmonds cycle contraction:

        AdjustedScore(h -> d) = Score(h -> d) + CycleScore - Score(old_incoming_edge(d))
    """
    cycle_set = set(cycle)
    cycle_total = sum(score[arc] for arc in selected_cycle_arcs)
    incoming_cycle_arc = {dep: (head, dep) for head, dep in selected_cycle_arcs}

    rows = []
    for (head, dep), original_score in score.items():
        if dep in cycle_set and head not in cycle_set:
            removed_arc = incoming_cycle_arc[dep]
            removed_score = score[removed_arc]
            adjusted_score = original_score + cycle_total - removed_score
            rows.append({
                "outside_head": head,
                "cycle_dependent": dep,
                "original_score": original_score,
                "removed_cycle_arc": f"{removed_arc[0]}->{removed_arc[1]}",
                "removed_score": removed_score,
                "cycle_total": cycle_total,
                "adjusted_score": adjusted_score,
            })

    df = pd.DataFrame(rows).sort_values(
        ["cycle_dependent", "outside_head"]
    ).reset_index(drop=True)
    return df


selected_cycle_arcs = [
    ("B","A"),
    ("A","B")
]

entering_cycle_table(
    cycle_score,
    ["A", "B"],
    selected_cycle_arcs
)

,outside_head,cycle_dependent,original_score,removed_cycle_arc,removed_score,cycle_total,adjusted_score
0,C,A,2,B->A,10,20,12
1,ROOT,A,5,B->A,10,20,15
2,C,B,2,A->B,10,20,12
3,ROOT,B,4,A->B,10,20,14


**Task 4 interpretation:**

The table (also including the symmetric `C -> A` / `C -> B` edges for completeness) is:

| outside_head | cycle_dependent | original_score | removed_cycle_arc | removed_score | cycle_total | adjusted_score |
|---|---|---:|---|---:|---:|---:|
| C | A | 2 | B->A | 10 | 20 | 12 |
| ROOT | A | 5 | B->A | 10 | 20 | **15** |
| C | B | 2 | A->B | 10 | 20 | 12 |
| ROOT | B | 4 | A->B | 10 | 20 | 14 |

**Which candidate edge should be selected to enter the contracted cycle?**
`ROOT -> A`, with the highest adjusted score of 15.

**Which original cycle edge is removed when the contracted tree is expanded back?**
Because the winning edge `ROOT -> A` enters the cycle at node `A`, node `A`'s *original* selected incoming edge inside the cycle, `B -> A`, is the one that gets discarded. The other cycle edge, `A -> B`, is kept, since `B` still needs an incoming arc and no better external edge won for `B`. So the final expanded tree contains `ROOT -> A` and `A -> B` (replacing the cyclic pair `A -> B`, `B -> A` with a single incoming arc into the cycle plus the rest of the cycle's internal edges).

## Task 5: Implement the Chu-Liu/Edmonds MST Parser (4 marks)

In this task, you will implement the **Chu-Liu/Edmonds algorithm from scratch** for finding the maximum-scoring rooted directed spanning tree.

Do **not** use:

- `networkx.maximum_spanning_arborescence`
- any other library implementation of Chu-Liu/Edmonds

You may use Python data structures such as dictionaries, lists, and sets.

### Algorithm

Your implementation should follow these steps:

#### Step 1: Select the best incoming edge

For every non-root node, select the incoming edge with the highest score.

#### Step 2: Check for cycles

If the selected edges form a valid rooted tree, return them.

If a directed cycle exists, continue.

#### Step 3: Identify the cycle

Find one directed cycle:

$$
C=\{v_1,v_2,\ldots,v_k\}
$$

#### Step 4: Contract the cycle

Replace all nodes in the cycle by a single supernode.

For an edge entering the cycle:

$$
u\rightarrow v,\qquad v\in C
$$

use:

$$
w'(u\rightarrow C)
=
w(u\rightarrow v)
+
W_C
-
w(parent(v)\rightarrow v)
$$

where $W_C$ is the total score of the selected cycle edges.

For an edge leaving the cycle:

$$
v\rightarrow u,\qquad v\in C
$$

keep the maximum-scoring corresponding outgoing edge.

#### Step 5: Recursively solve the contracted graph

Run the same procedure on the contracted graph.

#### Step 6: Expand the cycle

Use the selected incoming edge to determine which cycle edge must be removed.

Restore the remaining cycle edges and return the final maximum spanning arborescence.

### Required functions

Complete:

```python
chu_liu_edmonds()

In [6]:

def select_best_incoming(nodes, edges, root="ROOT"):
    """
    Select the highest-scoring incoming edge for every non-root node.

    edges:
        dictionary {(head, dependent): score}

    Returns:
        dict {dependent: (head, score)}
    """
    best = {}
    for dep in nodes:
        if dep == root:
            continue
        candidates = [
            (head, edges[(head, dep)])
            for head in nodes
            if head != dep and (head, dep) in edges
        ]
        if not candidates:
            continue
        best[dep] = max(candidates, key=lambda pair: pair[1])
    return best


def detect_cycle(arcs, root="ROOT"):
    """
    Return one directed cycle from arcs (a list of (head, dep) tuples).
    Return None if no cycle exists.
    """
    head_of = {dep: head for head, dep in arcs}

    visited = set()
    for start in head_of:
        if start in visited:
            continue
        path = []
        seen = {}
        current = start
        while current in head_of and current not in visited:
            if current in seen:
                idx = path.index(current)
                return path[idx:]
            seen[current] = True
            path.append(current)
            current = head_of[current]
        visited.update(path)
    return None


def contract_cycle(nodes, edges, selected_arcs, cycle, root="ROOT"):
    """
    Contract the detected cycle into a single supernode.

    selected_arcs:
        dict {dependent: (head, score)} -- the current best-incoming
        selection (from select_best_incoming) for THIS graph, used to look
        up each cycle node's currently-selected incoming edge and score.

    Return:
        contracted_nodes, contracted_edges, mapping_information

    mapping_information records, for every contracted edge, which
    original edge it came from, so expand_cycle() can restore the
    original graph afterwards.
    """
    cycle_set = set(cycle)
    supernode = "SUPER(" + ",".join(cycle) + ")"
    cycle_total = sum(selected_arcs[dep][1] for dep in cycle)

    contracted_nodes = [n for n in nodes if n not in cycle_set] + [supernode]
    contracted_edges = {}

    entering_source = {}   # (u, supernode) -> original (u, v) edge, v in cycle
    leaving_source = {}    # (supernode, w) -> original (v, w) edge, v in cycle

    for (u, v), s in edges.items():
        u_in, v_in = u in cycle_set, v in cycle_set

        if u_in and v_in:
            # Internal cycle edge -- irrelevant once contracted.
            continue

        elif v_in and not u_in:
            # Edge entering the cycle: u -> v, v in C.
            old_incoming_score = selected_arcs[v][1]
            adjusted = s + cycle_total - old_incoming_score
            key = (u, supernode)
            if key not in contracted_edges or adjusted > contracted_edges[key]:
                contracted_edges[key] = adjusted
                entering_source[key] = (u, v)

        elif u_in and not v_in:
            # Edge leaving the cycle: v -> w, v in C. Keep the max-scoring
            # outgoing edge for each external destination w.
            key = (supernode, v)
            if key not in contracted_edges or s > contracted_edges[key]:
                contracted_edges[key] = s
                leaving_source[key] = (u, v)

        else:
            # Neither endpoint touches the cycle -- unchanged.
            contracted_edges[(u, v)] = s

    mapping_information = {
        "cycle": list(cycle),
        "supernode": supernode,
        "entering_source": entering_source,
        "leaving_source": leaving_source,
        "selected_arcs": selected_arcs,
        "cycle_total": cycle_total,
    }

    return contracted_nodes, contracted_edges, mapping_information


def expand_cycle(
    selected_contracted_arcs,
    cycle,
    mapping_information,
    original_selected_arcs
):
    """
    Expand the contracted solution back to the original graph.

    selected_contracted_arcs: list of (head, dep) tuples -- the solution
        found recursively on the contracted graph.

    The incoming edge selected for the supernode determines which cycle
    edge must be removed; every other cycle-internal edge is restored.
    """
    supernode = mapping_information["supernode"]
    entering_source = mapping_information["entering_source"]
    leaving_source = mapping_information["leaving_source"]
    selected_arcs = mapping_information["selected_arcs"]

    final_arcs = []
    incoming_edge_to_cycle = None

    for head, dep in selected_contracted_arcs:
        if dep == supernode:
            # The edge that enters the contracted cycle.
            orig_u, orig_v = entering_source[(head, supernode)]
            incoming_edge_to_cycle = (orig_u, orig_v)
            final_arcs.append(incoming_edge_to_cycle)
        elif head == supernode:
            # An edge leaving the cycle.
            orig_u, orig_v = leaving_source[(supernode, dep)]
            final_arcs.append((orig_u, orig_v))
        else:
            final_arcs.append((head, dep))

    # Restore every cycle-internal edge except the one whose dependent now
    # receives the external incoming edge (that arc is the one "broken"
    # by the winning external edge).
    removed_node = incoming_edge_to_cycle[1] if incoming_edge_to_cycle else None
    for node in cycle:
        if node == removed_node:
            continue
        head, _score = selected_arcs[node]
        final_arcs.append((head, node))

    return final_arcs


def chu_liu_edmonds(tokens, score, root="ROOT"):
    """
    Chu-Liu/Edmonds maximum spanning arborescence, implemented from
    scratch (no networkx).

    Return:
        total_score, final_arcs
    """

    def _solve(nodes, edges):
        # Step 1: select the highest-scoring incoming edge for every
        # non-root node.
        best = select_best_incoming(nodes, edges, root)
        arcs = [(head, dep) for dep, (head, _s) in best.items()]

        # Step 2 & 3: check whether the selected edges contain a cycle.
        cycle = detect_cycle(arcs, root)

        # If there is no cycle, the greedy selection IS the arborescence.
        if cycle is None:
            return arcs

        # Step 4: contract the cycle into a supernode.
        c_nodes, c_edges, mapping = contract_cycle(nodes, edges, best, cycle, root)

        # Step 5: recursively solve the (smaller) contracted graph.
        contracted_solution = _solve(c_nodes, c_edges)

        # Step 6: expand the cycle back into the original graph.
        return expand_cycle(contracted_solution, cycle, mapping, best)

    final_arcs = _solve(list(tokens), dict(score))
    total_score = sum(score[a] for a in final_arcs)
    return total_score, final_arcs


# Test on the supplied graph
cle_score, cle_arcs = chu_liu_edmonds(tokens, score)

print("Chu-Liu/Edmonds score:", cle_score)
print("Chu-Liu/Edmonds arcs:", cle_arcs)

print(
    "Matches exhaustive MST score?",
    cle_score == best_score
)

Chu-Liu/Edmonds score: 26
Chu-Liu/Edmonds arcs: [('analyze', 'Researchers'), ('ROOT', 'analyze'), ('analyze', 'data'), ('analyze', 'carefully')]
Matches exhaustive MST score? True


## Task 6: Inference-Based Learning with Structured Perceptron (5 marks)

In this task, you will train the MST dependency parser using **inference-based structured perceptron learning**.

The parser uses the current weight vector to score every possible dependency tree and selects the highest-scoring tree:

$$
\boxed{
G_{pred}
=
\arg\max_{G\in T(x)}
\mathbf{w}\cdot\Phi(x,G)
}
$$

where:

- $x$ = input sentence
- $T(x)$ = set of valid dependency trees for the sentence
- $\mathbf{w}$ = current feature-weight vector
- $\Phi(x,G)$ = feature vector of dependency tree $G$

The predicted tree is then compared with the gold tree.

If:

$$
G_{pred}\neq G_{gold}
$$

update the weights using:

$$
\boxed{
\mathbf{w}_{new}
=
\mathbf{w}_{old}
+
\Phi(x,G_{gold})
-
\Phi(x,G_{pred})
}
$$

Equivalently, for each feature:

$$
\boxed{
w_{new}[f]
=
w_{old}[f]
+
\Phi(x,G_{gold})[f]
-
\Phi(x,G_{pred})[f]
}
$$

### Training procedure

Your training loop must perform the following steps:

1. Initialize the feature weights.
2. Take one training sentence and its gold dependency tree.
3. Compute the score of candidate dependency arcs using the current weights.
4. Run your **own Chu-Liu/Edmonds implementation from Task 5** to obtain the predicted tree.
5. Compute the feature vector of the gold tree.
6. Compute the feature vector of the predicted tree.
7. Update the weights using the structured perceptron rule.
8. Repeat for all training examples.
9. Repeat the complete process for multiple epochs.

### Important

The training procedure must call your implementation from **Task 5**.

Do not use:

```python
networkx.maximum_spanning_arborescence()

In [7]:

from collections import Counter

# ---------------------------------------------------------
# Training data
# ---------------------------------------------------------

training_data = [

    {
        "tokens": ["ROOT", "John", "saw", "Mary"],
        "pos": {
            "ROOT": "ROOT",
            "John": "NOUN",
            "saw": "VERB",
            "Mary": "NOUN"
        },
        "gold": [
            ("ROOT", "saw"),
            ("saw", "John"),
            ("saw", "Mary")
        ]
    },

    {
        "tokens": ["ROOT", "Dogs", "chase", "cats"],
        "pos": {
            "ROOT": "ROOT",
            "Dogs": "NOUN",
            "chase": "VERB",
            "cats": "NOUN"
        },
        "gold": [
            ("ROOT", "chase"),
            ("chase", "Dogs"),
            ("chase", "cats")
        ]
    },

    {
        "tokens": ["ROOT", "Alice", "likes", "Bob"],
        "pos": {
            "ROOT": "ROOT",
            "Alice": "NOUN",
            "likes": "VERB",
            "Bob": "NOUN"
        },
        "gold": [
            ("ROOT", "likes"),
            ("likes", "Alice"),
            ("likes", "Bob")
        ]
    }
]


# ---------------------------------------------------------
# Feature extraction
# ---------------------------------------------------------

def arc_features(tokens, pos, head, dep):
    """
    Return a list of feature names for one dependency arc.

    Required feature templates:
      1. head word
      2. dependent word
      3. head POS
      4. dependent POS
      5. direction
      6. distance
      7. head-dependent word pair
    """
    hi = tokens.index(head)
    di = tokens.index(dep)
    direction = "RIGHT" if hi < di else "LEFT"
    distance = abs(hi - di)

    return [
        f"HEAD_WORD={head}",
        f"DEP_WORD={dep}",
        f"HEAD_POS={pos[head]}",
        f"DEP_POS={pos[dep]}",
        f"DIRECTION={direction}",
        f"DISTANCE={distance}",
        f"PAIR={head}->{dep}",
    ]


def tree_feature_vector(tokens, pos, tree):
    """
    Sum the feature vectors of all arcs in a dependency tree.

    Return a Counter or dictionary:
        feature -> count
    """
    vec = Counter()
    for head, dep in tree:
        for f in arc_features(tokens, pos, head, dep):
            vec[f] += 1
    return vec


# ---------------------------------------------------------
# Arc and tree scoring
# ---------------------------------------------------------

def score_arc(tokens, pos, head, dep, weights):
    """
    Score one dependency arc:

        score(h -> d) = w . f(h -> d)

    """
    return sum(weights[f] for f in arc_features(tokens, pos, head, dep))


def build_score_graph(tokens, pos, weights):
    """
    Construct the complete directed graph of candidate
    dependency arcs.

    Return:
        {(head, dependent): score}
    """
    graph = {}
    for head in tokens:
        for dep in tokens:
            if head == dep or dep == "ROOT":
                continue
            graph[(head, dep)] = score_arc(tokens, pos, head, dep, weights)
    return graph


def score_tree(tokens, pos, tree, weights):
    """
    Return the total score of a dependency tree.
    """
    return sum(score_arc(tokens, pos, head, dep, weights) for head, dep in tree)


# ---------------------------------------------------------
# Structured perceptron update
# ---------------------------------------------------------

def perceptron_update(
    weights,
    gold_features,
    predicted_features,
    learning_rate=1.0
):
    """
    Structured perceptron update:

        w_new =
        w_old
        + eta * Phi(gold)
        - eta * Phi(predicted)
    """
    new_weights = Counter(weights)
    for f, c in gold_features.items():
        new_weights[f] += learning_rate * c
    for f, c in predicted_features.items():
        new_weights[f] -= learning_rate * c
    return new_weights


# ---------------------------------------------------------
# Inference-based training
# ---------------------------------------------------------

def train_perceptron(
    training_data,
    num_epochs=5,
    learning_rate=1.0
):
    """
    Train the MST parser using inference-based
    structured perceptron learning.

    For every epoch:

        1. Infer predicted tree using Task 5.
        2. Compare with gold tree.
        3. Update weights if prediction is incorrect.
        4. Record number of mistakes.
    """

    # Initialize all weights to zero.
    weights = Counter()

    history = []

    for epoch in range(num_epochs):

        mistakes = 0

        for example in training_data:

            tokens = example["tokens"]
            pos = example["pos"]
            gold_tree = example["gold"]

            # ---------------------------------------------
            # Build arc scores using current weights.
            # ---------------------------------------------

            score_graph = build_score_graph(
                tokens,
                pos,
                weights
            )

            # ---------------------------------------------
            # Run YOUR Task 5 implementation.
            # ---------------------------------------------

            predicted_score, predicted_tree = chu_liu_edmonds(
                tokens,
                score_graph
            )

            # ---------------------------------------------
            # Compare predicted and gold trees.
            # ---------------------------------------------

            if set(predicted_tree) != set(gold_tree):

                mistakes += 1

                # -----------------------------------------
                # Compute gold feature vector.
                # -----------------------------------------

                gold_features = tree_feature_vector(
                    tokens,
                    pos,
                    gold_tree
                )

                # -----------------------------------------
                # Compute predicted feature vector.
                # -----------------------------------------

                predicted_features = tree_feature_vector(
                    tokens,
                    pos,
                    predicted_tree
                )

                # -----------------------------------------
                # Update weights.
                # -----------------------------------------

                weights = perceptron_update(
                    weights,
                    gold_features,
                    predicted_features,
                    learning_rate
                )

        history.append(mistakes)

        print(
            f"Epoch {epoch + 1}: "
            f"{mistakes} mistake(s)"
        )

    return weights, history

In [8]:
# Run the complete inference-based training

final_weights, training_history = train_perceptron(
    training_data,
    num_epochs=5,
    learning_rate=1.0
)

print("\nTraining history:")
for epoch, mistakes in enumerate(training_history, start=1):
    print(
        f"Epoch {epoch}: "
        f"{mistakes} mistake(s)"
    )

print("\nLearned weights:")
for feature, weight in sorted(final_weights.items()):
    print(f"{feature}: {weight}")

Epoch 1: 1 mistake(s)
Epoch 2: 0 mistake(s)
Epoch 3: 0 mistake(s)
Epoch 4: 0 mistake(s)
Epoch 5: 0 mistake(s)

Training history:
Epoch 1: 1 mistake(s)
Epoch 2: 0 mistake(s)
Epoch 3: 0 mistake(s)
Epoch 4: 0 mistake(s)
Epoch 5: 0 mistake(s)

Learned weights:
DEP_POS=NOUN: 0.0
DEP_POS=VERB: 0.0
DEP_WORD=John: 0.0
DEP_WORD=Mary: 0.0
DEP_WORD=saw: 0.0
DIRECTION=LEFT: 1.0
DIRECTION=RIGHT: -1.0
DISTANCE=1: 1.0
DISTANCE=2: 0.0
DISTANCE=3: -1.0
HEAD_POS=ROOT: -2.0
HEAD_POS=VERB: 2.0
HEAD_WORD=ROOT: -2.0
HEAD_WORD=saw: 2.0
PAIR=ROOT->John: -1.0
PAIR=ROOT->Mary: -1.0
PAIR=ROOT->saw: 0.0
PAIR=saw->John: 1.0
PAIR=saw->Mary: 1.0


## Task 6: Training Analysis

### Q1. Prediction Errors

For each epoch, report the number of incorrectly predicted trees.

| Epoch | Number of Mistakes |
|------:|-------------------:|
| 1 | 1 |
| 2 | 0 |
| 3 | 0 |
| 4 | 0 |
| 5 | 0 |

### Q2. Weight Updates

Two features whose weights increased during training:

- **`HEAD_WORD=saw` (0 → +2.0)** and **`HEAD_POS=VERB` (0 → +2.0)**: these fire on every gold arc `saw -> John` and `saw -> Mary` but not on the wrong prediction, which used `ROOT` as head instead of `saw`. Since the perceptron rule adds `Phi(gold)` and subtracts `Phi(predicted)`, any feature present in the gold tree's arcs but absent from the predicted tree's arcs gets a positive net update, making the model favour `saw` (a verb) as a head next time.
- **`DIRECTION=LEFT` (0 → +1.0)** and **`DISTANCE=1` (0 → +1.0)**: `saw -> John` is a right-to-left arc at distance 1 in the gold tree, while the mistaken prediction `ROOT -> John` is a left-to-right arc at distance 1. `DIRECTION=LEFT` only appears in the gold arc, so it gets pushed up.

### Q3. Negative Updates

Two features whose weights decreased during training:

- **`HEAD_WORD=ROOT` (0 → -2.0)** and **`HEAD_POS=ROOT` (0 → -2.0)**: the wrong prediction used `ROOT` as the head for `John`, `saw`, and `Mary`. `ROOT` never appears as a head in the gold tree except for `ROOT -> saw`, so these features fire more often in the predicted (wrong) tree than in the gold tree, and the update `+Phi(gold) - Phi(predicted)` pushes their weight down — discouraging `ROOT` from being (mis)used as a head of ordinary words.
- **`PAIR=ROOT->John` (0 → -1.0)** and **`PAIR=ROOT->Mary` (0 → -1.0)**: these exact head-dependent pairs occur only in the wrong prediction (`ROOT -> John`, `ROOT -> Mary`), never in the gold tree, so they are pure "predicted-only" features and receive a full negative update.

### Q4. Effect of the Perceptron Update

$$
\mathbf{w}_{new}=\mathbf{w}_{old}+\Phi(x,G_{gold})-\Phi(x,G_{pred})
$$

Adding `Phi(gold)` increases the weight of every feature that appears on an arc of the gold tree, so those arcs will score higher the next time inference is run. Subtracting `Phi(pred)` decreases the weight of every feature that appears on an arc of the (wrong) predicted tree, so those arcs will score lower next time. Features shared by both trees (e.g. `ROOT -> saw`, which is correct in both) get `+1` and `-1` for the same feature, cancelling out — the update only changes the *relative* score between the gold structure and the specific mistake that was made. Since `score(G) = w . Phi(x, G)` is linear in `w`, this shift directly raises `w . Phi(gold)` and lowers `w . Phi(pred)` by the same amount, so `Score(G_gold) - Score(G_pred)` strictly increases after the update. That makes it more likely that `G_gold` (or a tree very similar to it) will win the `argmax` in the next round of Chu-Liu/Edmonds inference.

### Q5. Training Behaviour

Yes — the number of mistakes dropped from 1 (epoch 1) to 0 for every subsequent epoch. This happened very quickly (after a single update) because:

- All three training sentences share the same template (`NOUN VERB NOUN`, i.e. `ROOT SUBJ VERB OBJ`), so the feature templates that matter here (`HEAD_POS`, `DIRECTION`, `DISTANCE`) generalize across sentences even though the specific words differ.
- After the one mistake on `"ROOT John saw Mary"`, the update already pushed `HEAD_POS=VERB`, `DIRECTION=LEFT`, and `DISTANCE=1` upward and `HEAD_POS=ROOT` downward — exactly the structural cues needed to correctly parse `"Dogs chase cats"` and `"Alice likes Bob"` as well, even though the model had never seen those sentences' specific words before. Once the weights correctly encode "the verb is usually the head of its left/right neighbouring nouns," every subsequent sentence (and subsequent epoch) is parsed correctly and no further updates occur.